# The pipeline, step by step

This notebook runs each step of the ingest pipeline on the golden filing (Maroc Telecom, Rapport
financier annuel 2025) with the same functions the app uses, and shows what each step produces.
New steps are tried here first, then moved into `bloomberg_invest`.

| Step | Code | Status |
| --- | --- | --- |
| 1. Parse | read from the app: `bloomberg ingest` | in the app (#20) |
| 2. Read blocks | `DoclingParser.read_blocks` | in the app (#21 PR A) |
| 3. Chunk | `domain/chunking.py` | in the app (#21 PR A) |
| 4. Embed | sentence-transformers, below | **prototype** for #21 PR B |
| 5. Search | full-text, vector and hybrid, below | **prototype** for #21 PR B |

It reads the app's own parse of the filing, so it shows exactly what the app holds. Start the
stack and ingest the golden PDF first (README), then run it from `backend/`:

```bash
uv run --env-file ../.env --extra ingest --group notebook jupyter lab notebooks/pipeline.ipynb
```

Outputs are stripped before commit (`nbstripout`): they show pages of the AMMC report, which is
not redistributed. It only reads from Neo4j.

In [ ]:
import csv
import hashlib
import json
import math
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypdfium2 as pdfium
from IPython.display import Markdown, display
from matplotlib.patches import Rectangle
from neo4j import GraphDatabase, NotificationDisabledClassification

from bloomberg_invest.adapters.parsing.docling_parser import DoclingParser
from bloomberg_invest.adapters.store.neo4j_store import Neo4jStore
from bloomberg_invest.adapters.tokens import TiktokenCounter
from bloomberg_invest.domain.chunking import SectionRef, chunk
from bloomberg_invest.domain.indexing import index_text, is_search_unit
from bloomberg_invest.settings import Settings

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd().parent
DATA = ROOT / "data"
PDF = DATA / "inbox" / "Maroc_Telecom_RFA_2025.pdf"
WORK = DATA / "notebook"  # gitignored, like all of data/
WORK.mkdir(parents=True, exist_ok=True)
CAP = 2000  # CHUNK_CAP

pd.set_option("display.max_colwidth", 90)
assert PDF.exists(), f"put the golden PDF at {PDF} (golden.md gives the link and SHA-256)"
print(PDF, hashlib.sha256(PDF.read_bytes()).hexdigest()[:12])

## 1. Parse

The app parses each filing once, with Docling in its Docker image, and saves Docling's output with
the parse Run. This step finds the golden filing in the app's graph by its SHA-256 and reads that
Run, so every step below starts from the app's own parse. The ingest image installs the fonts the
report uses but doesn't embed; without them, Docling misreads whole pages (#36).

In [ ]:
settings = Settings.from_env()
sha256 = hashlib.sha256(PDF.read_bytes()).hexdigest()
with GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password),
    notifications_disabled_classifications=[NotificationDisabledClassification.UNRECOGNIZED],
) as driver:
    store = Neo4jStore(driver)
    details = (store.get_document(d.id) for d in store.list_documents())
    detail = next((d for d in details if d.sha256 == sha256), None)
assert detail, "ingest the golden PDF into the app first (README: docker compose run --rm ingest)"
assert detail.run.output, "the parse run saved no output; re-ingest"

parser = DoclingParser()
output = DATA / detail.run.output
run = detail.run
print(f"document {detail.id}, {detail.page_count} pages")
print(
    f"parse run {run.id}: {run.tool} {run.tool_version} {run.config}, finished {run.finished_at:%Y-%m-%d %H:%M}"
)
print(output.relative_to(DATA))

### The section outline

In [ ]:
def walk(nodes, depth=0):
    for n in nodes:
        yield depth, n
        yield from walk(n.children, depth + 1)


outline = list(walk(detail.sections))
print(
    f"{len(outline)} sections; by level: {dict(sorted(Counter(n.level for _, n in outline).items()))}"
)
for depth, n in outline[:45]:
    print(f"{'    ' * depth}{n.title[:70]}  (p. {n.page})")

### A page with its headings

Each Section keeps its heading's box in PDF points from the top-left. The viewer draws highlights
the same way.

In [ ]:
pdf = pdfium.PdfDocument(PDF)
sizes = {p.number: (p.width, p.height) for p in detail.pages}


def show_page(number, boxes=(), title=None, scale=1.4):
    """Draw a page with boxes: (left, top, right, bottom, colour, label)."""
    image = pdf[number - 1].render(scale=scale).to_pil()
    _, ax = plt.subplots(figsize=(9, 9 * image.height / image.width))
    ax.imshow(image)
    for left, top, right, bottom, colour, label in boxes:
        ax.add_patch(
            Rectangle(
                (left * scale, top * scale),
                (right - left) * scale,
                (bottom - top) * scale,
                fill=False,
                lw=1.6,
                ec=colour,
            )
        )
        if label:  # in the right margin, clear of the page's own text
            ax.text(right * scale + 4, top * scale + 8, label, color=colour, fontsize=7)
    ax.set_title(title or f"Page {number}")
    ax.axis("off")
    plt.show()


PAGE = 29
show_page(PAGE, [(*n.bbox, "crimson", None) for _, n in outline if n.page == PAGE and n.bbox])

## 2. Read blocks

The chunker reads the saved Docling output back as headings, paragraphs and tables in reading
order. A paragraph keeps every box it covers; `numbered` marks list items and numbered points.

In [ ]:
blocks = parser.read_blocks(output)
frame = pd.DataFrame(
    [
        {
            "kind": b.kind,
            "page": b.page,
            "pages": len(b.pages),
            "numbered": b.numbered,
            "columns": b.columns,
            "rows": len(b.rows),
            "text": (b.text or b.header)[:90],
        }
        for b in blocks
    ]
)
print(frame.kind.value_counts().to_dict())
frame.head(25)

## 3. Chunk

The rules (#19 R2): a table is one Chunk, joined when it continues on the next page; a section's
paragraphs form one Chunk under the cap; a longer section splits at numbered points, otherwise
evenly; a paragraph is never split. Token counts use `cl100k_base`.

In [ ]:
count = TiktokenCounter()
refs = [SectionRef(id=n.id, title=n.title, page=n.page) for _, n in outline]
titles = {r.id: r.title for r in refs}
chunks = chunk(blocks, refs, CAP, count)

table = pd.DataFrame(
    [
        {
            "index": c.index,
            "kind": c.kind,
            "section_id": c.section_id,
            "section": titles.get(c.section_id, "")[:45],
            "pages": f"{c.pages[0]}-{c.pages[-1]}" if len(c.pages) > 1 else str(c.pages[0]),
            "tokens": c.token_count,
            "paragraphs": len(c.paragraphs),
            "cites": ", ".join(titles[s][:12] for s in c.refers_to),
        }
        for c in chunks
    ]
)
print(
    f"{len(chunks)} chunks ({(table.kind == 'table').sum()} tables), "
    f"{sum(len(c.paragraphs) for c in chunks)} paragraphs"
)
table.drop(columns="section_id").head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, kind in zip(axes, ["text", "table"]):
    ax.hist(table[table.kind == kind].tokens, bins=40, color="#4c72b0")
    ax.axvline(CAP, color="crimson", ls="--", label=f"cap {CAP}")
    ax.set_title(f"{kind} chunks: tokens")
    ax.legend()
plt.show()
print("text chunks over the cap (single paragraphs only):")
table[(table.kind == "text") & (table.tokens > CAP)]

### Sections split by the cap, and tables joined across pages

golden.md lists the true multi-page tables (§5): the key audit matters on pp. 74-75 and 86-87.

In [ ]:
# Group by Section id: titles repeat ("État de résultat global" is several Sections).
text_chunks = table[table.kind == "text"]
split = text_chunks.groupby("section_id").filter(lambda g: len(g) > 1)
if split.empty:
    print(f"No section's text is over the cap of {CAP} tokens in this parse.")
else:
    display(
        split.groupby(["section_id", "section"]).agg(
            chunks=("index", "count"), tokens=("tokens", "sum")
        )
    )
joined = table[(table.kind == "table") & table.pages.str.contains("-")]
joined

In [ ]:
c = chunks[int(joined.iloc[0]["index"])] if len(joined) else None
display(Markdown(c.text[:3000] if c else "No table was joined."))

### The paragraphs of one page, coloured by Chunk

In [ ]:
PAGE = 10
colours = plt.cm.tab10.colors
on_page = [(c, p) for c in chunks for p in c.paragraphs if PAGE in p.pages]
order = {c.index: i for i, c in enumerate(dict.fromkeys(c for c, _ in on_page))}
boxes, labelled = [], set()
for c, p in on_page:
    for pg, left, top, right, bottom in p.boxes:
        if pg == PAGE:
            label = None if c.index in labelled else f"chunk {c.index}"
            labelled.add(c.index)
            boxes.append((left, top, right, bottom, colours[order[c.index] % 10], label))
show_page(PAGE, boxes)

### Notes a Chunk cites

`REFERS_TO` links a Chunk to the NOTE it cites. golden.md §6 lists p. 29's "(cf. note 14)",
"(cf. note 3)", "(cf. note 8)" and "la note 2".

In [ ]:
table[(table.cites != "") & table.pages.str.startswith("29")]

## 4. Embed (prototype for #21 PR B)

The search unit is the Paragraph, headings aside: a heading's title already leads the
Paragraphs under it. Each is indexed as `index_text` gives it: its section path and page, then
its text, a table one line per row. Two open models from the research, run locally:

| Model | Licence | Dims | Query prefix |
| --- | --- | --- | --- |
| `Qwen/Qwen3-Embedding-0.6B` | Apache 2.0 | 1,024 | `query` prompt |
| `BAAI/bge-m3` | MIT | 1,024 | none |

Vectors are cached in `data/notebook/` by model and indexed text, so a rerun is instant.

In [ ]:
from sentence_transformers import SentenceTransformer

MODELS = {
    "qwen3-0.6b": {"name": "Qwen/Qwen3-Embedding-0.6B", "query_prompt": "query"},
    "bge-m3": {"name": "BAAI/bge-m3", "query_prompt": None},
}
paragraphs = [(c, p) for c in chunks for p in c.paragraphs]
texts = [p.text for _, p in paragraphs]  # what readers see
parent = {n.id: None for _, n in outline}
for _, n in outline:
    for child in n.children:
        parent[child.id] = n.id


def lineage(section_id):
    """A Section and every Section above it."""
    while section_id:
        yield section_id
        section_id = parent[section_id]


units = np.array([is_search_unit(p) for _, p in paragraphs])
indexed = [
    index_text(p, [titles[s] for s in reversed(list(lineage(c.section_id)))]) for c, p in paragraphs
]
key = hashlib.sha256("\n\0".join(indexed).encode()).hexdigest()[:12]
loaded, vectors, timings = {}, {}, {}

for label, spec in MODELS.items():
    model = loaded[label] = SentenceTransformer(spec["name"])
    model.max_seq_length = (
        1024  # long tables are cut for the vector; their full text stays searchable
    )
    cache = WORK / f"embeddings-{label}-{key}.npy"
    started = time.perf_counter()
    if cache.exists():
        vectors[label] = np.load(cache)
    else:
        vectors[label] = model.encode(
            indexed, batch_size=16, normalize_embeddings=True, show_progress_bar=True
        )
        np.save(cache, vectors[label])
    timings[label] = time.perf_counter() - started
    print(f"{label}: {vectors[label].shape} in {timings[label]:.0f}s on {model.device}")

## 5. Search (prototype for #21 PR B)

- **Full-text:** BM25 over the indexed text, lower-cased with accents folded: the stand-in for
  Neo4j's `standard-folding` index.
- **Vector:** cosine similarity with each model.
- **Hybrid:** reciprocal rank fusion of the two, as `HybridCypherRetriever` combines them.
- Headings are left out of every ranking.
- **Hybrid-fr:** the question goes to full-text in French too (below), fused with the English
  vector search.
- **Rerank:** the best 50 of hybrid-fr, reordered by a cross-encoder that reads the question
  with each Paragraph.

In [ ]:
def fold(text):
    text = unicodedata.normalize("NFKD", text.lower())
    return re.findall(r"\w+", "".join(ch for ch in text if not unicodedata.combining(ch)))


docs = [fold(t) for t in indexed]
df = Counter(term for d in docs for term in set(d))
avg = sum(map(len, docs)) / len(docs)


def bm25(query, k1=1.2, b=0.75):
    terms = fold(query)
    scores = np.zeros(len(docs))
    for i, d in enumerate(docs):
        tf = Counter(d)
        for t in terms:
            if tf[t]:
                idf = math.log(1 + (len(docs) - df[t] + 0.5) / (df[t] + 0.5))
                scores[i] += idf * tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(d) / avg))
    return scores


def vector(query, label):
    spec, model = MODELS[label], loaded[label]
    kwargs = {"prompt_name": spec["query_prompt"]} if spec["query_prompt"] else {}
    q = model.encode([query], normalize_embeddings=True, **kwargs)[0]
    return vectors[label] @ q


def ranks(scores):
    """Search units, best first; headings are left out."""
    order = np.argsort(-scores, kind="stable")
    return order[units[order]]


def fuse(*rankings, k=60):
    """Reciprocal rank fusion of several score arrays."""
    fused = np.zeros(len(texts))
    for scores in rankings:
        for rank, i in enumerate(ranks(scores)):
            fused[i] += 1 / (k + rank + 1)
    return fused


def hybrid(query, label):
    return fuse(bm25(query), vector(query, label))


def top(scores, n=5):
    return [(i, paragraphs[i][1].page, texts[i][:110]) for i in ranks(scores)[:n]]

### English questions, French report

Full-text can't match "revenue" to "chiffre d'affaires". An LLM (the Azure OpenAI deployment the
parser bake-off uses) gives each question in French and the French terms its answer likely uses;
full-text searches both. Answers are cached in `data/notebook/translations.json`, so the key is
needed only for a question not seen before: run the notebook with
`--env-file ~/.config/bloomberg-invest/bakeoff.env` as well.

In [ ]:
import os

from openai import OpenAI

TRANSLATE = """You search a French annual financial report filed with the AMMC (Morocco) for the
answer to a question. Give the question in French as the report would phrase it, and the French
terms most likely to appear in the passage that answers it: line items, note titles, table
labels. Answer in JSON: {"french": "...", "keywords": ["...", "..."]}."""
translations_path = WORK / "translations.json"
translations = json.loads(translations_path.read_text()) if translations_path.exists() else {}


def french(question):
    """The question in French and the French terms of its answer, from the cache or the LLM."""
    if question not in translations:
        assert "AZURE_OPENAI_API_KEY" in os.environ, "run with the bake-off env file (see above)"
        endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
        client = OpenAI(
            base_url=f"{endpoint}/openai/v1/", api_key=os.environ["AZURE_OPENAI_API_KEY"]
        )
        deployment = os.environ["AZURE_OPENAI_DEPLOYMENTS"].split(",")[0].strip()
        reply = client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": TRANSLATE},
                {"role": "user", "content": question},
            ],
            response_format={"type": "json_object"},
        )
        translations[question] = {
            "deployment": deployment,
            **json.loads(reply.choices[0].message.content),
        }
        translations_path.write_text(json.dumps(translations, ensure_ascii=False, indent=1))
    return translations[question]


def hybrid_fr(query, label):
    fr = french(query)
    return fuse(bm25(fr["french"]), bm25(" ".join(fr["keywords"])), vector(query, label))

### Rerankers

A cross-encoder reads the question and one Paragraph together, so it can judge an English question
against French text directly. Both run locally on the best 50 of hybrid-fr with qwen3-0.6b:

| Model | Licence | How it scores |
| --- | --- | --- |
| `BAAI/bge-reranker-v2-m3` | Apache 2.0 | a relevance score (sentence-transformers `CrossEncoder`) |
| `Qwen/Qwen3-Reranker-0.6B` | Apache 2.0 | the probability it answers "yes", with the model card's prompt |

In [ ]:
import torch
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
POOL = 50
DOC_CHARS = 4000  # long tables are cut for the reranker, as for the embedder


class Qwen3Reranker:
    """Qwen3-Reranker answers "yes" or "no" to whether a document meets the query; the score is
    P(yes). Prompt and scoring follow the model card."""

    PREFIX = (
        "<|im_start|>system\nJudge whether the Document meets the requirements based on the Query "
        'and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n'
        "<|im_start|>user\n"
    )
    SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    INSTRUCT = "Given a question about a company's annual financial report, judge whether the passage answers it"

    def __init__(self, name="Qwen/Qwen3-Reranker-0.6B"):
        self.tokenizer = AutoTokenizer.from_pretrained(name, padding_side="left")
        self.model = AutoModelForCausalLM.from_pretrained(name).eval().to(DEVICE)
        self.yes = self.tokenizer.convert_tokens_to_ids("yes")
        self.no = self.tokenizer.convert_tokens_to_ids("no")

    def predict(self, pairs, batch_size=8):
        scores = []
        for start in range(0, len(pairs), batch_size):
            prompts = [
                f"{self.PREFIX}<Instruct>: {self.INSTRUCT}\n<Query>: {q}\n<Document>: {d}{self.SUFFIX}"
                for q, d in pairs[start : start + batch_size]
            ]
            batch = self.tokenizer(prompts, padding=True, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                # Only the last position's logits: the full vocabulary at every position is GBs.
                logits = self.model(**batch, logits_to_keep=1).logits[:, -1, [self.no, self.yes]]
            scores += torch.softmax(logits.float(), dim=-1)[:, 1].tolist()
        return np.array(scores)


RERANKERS = {
    "bge-reranker-v2-m3": CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=1024, device=DEVICE),
    "qwen3-reranker-0.6b": Qwen3Reranker(),
}


def rerank(query, name, label="qwen3-0.6b"):
    """The best POOL of hybrid-fr, reordered by the reranker; the rest are left out."""
    pool = ranks(hybrid_fr(query, label))[:POOL]
    scores = np.full(len(texts), -np.inf)
    scores[pool] = RERANKERS[name].predict([(query, indexed[i][:DOC_CHARS]) for i in pool])
    return scores

### AC3: "dividende proposé" must return the p. 10 dividend paragraph in the top 5

In [ ]:
QUERY = "dividende proposé"
for name, scores in (
    [("full-text", bm25(QUERY))]
    + [(f"vector {m}", vector(QUERY, m)) for m in MODELS]
    + [(f"hybrid {m}", hybrid(QUERY, m)) for m in MODELS]
):
    print(f"\n{name}")
    for i, page, text in top(scores):
        print(f"  p. {page:>2}  {text}")

In [ ]:
best = ranks(hybrid(QUERY, "qwen3-0.6b"))[0]
c, p = paragraphs[best]
show_page(
    p.page,
    [(l, t, r, b, "orange", "top hit") for (pg, l, t, r, b) in p.boxes if pg == p.page],
    title=f'"{QUERY}": top hit, p. {p.page}',
)

### The golden questions

golden.md §8 gives each question. `evidence.csv` lists the places its citation names: a page, a
section, and a figure or short quote copied from that page. Each method is scored on its results:

| Score | What counts |
| --- | --- |
| page@5 | a top-5 Paragraph on a cited page (share of questions; the loose baseline) |
| section@5 | a top-5 Paragraph in the cited section or below it (share of evidence items) |
| evidence@5 | a top-5 Paragraph that holds the item on its page (share of evidence items) |
| MRR@10 | 1 / the rank of the first Paragraph holding any item, 0 past rank 10 |
| noise@5 | top-5 Paragraphs that are a page header or footer, a lone heading, or under 20 characters |

The questions are in English and the report in French, so the vector side must cross languages.

In [ ]:
QUESTIONS = {
    "Q1": ("What does Maroc Telecom forecast for 2026?", {10}),
    "Q2": (
        (
            "Which market risks does the group describe, and what is the impact of a 1% "
            "depreciation of the dirham against the euro?"
        ),
        {68, 69, 70},
    ),
    "Q3": (
        "What happened with the Wana Corporate dispute in 2025 and how did it affect results?",
        {68, 10, 15},
    ),
    "Q4": (
        (
            "With which related parties did Maroc Telecom transact in 2025, and how much revenue "
            "came from Etisalat?"
        ),
        {66, 67},
    ),
    "Q5": ("What was consolidated revenue (IFRS) for FY2025?", {24}),
    "Q6": (
        "What was consolidated net debt at 31 Dec 2025 and the net debt / EBITDA ratio?",
        {56, 9},
    ),
    "Q7": ("What was FY2025 net income attributable to the group?", {24, 9, 10}),
    "Q8": ("What was Maroc Telecom's company-only (CGNC) net income for 2025?", {81, 85}),
    "Q9": ("Who are Maroc Telecom's shareholders at 31 Dec 2025?", {27}),
    "Q10": (
        "Which fully consolidated subsidiaries are not wholly owned, and what is the interest?",
        {40, 63},
    ),
    "AC3": ("dividende proposé", {10}),
}

GOLDEN = ROOT / "golden" / "maroc-telecom-rfa-2025"
SPACE = r"[\s.]?"  # thousands separators: \s covers the no-break spaces French uses
FIGURE = re.compile(r"[+-]?\d[\d\s.,]*%?")


def pattern(number):
    """A number as written, matched with or without thousands separators. Tables read as flat
    text put numbers side by side, so a match may sit next to other digit groups."""
    groups = re.findall(r"\d+(?:,\d+)?%?", number)
    return re.compile(r"(?<![\d,])" + SPACE.join(map(re.escape, groups)) + r"(?!\d|,\d)")


def contains(text, number):
    return pattern(number).search(text) is not None


def folded(text):
    return " ".join(fold(text))


def holds(text, quote):
    """A figure is matched as a number; anything else word by word, case and accents folded."""
    return contains(text, quote) if FIGURE.fullmatch(quote) else folded(quote) in folded(text)


def cited_section(title, page):
    """The Section a citation names: the last one by its page whose title starts that way."""
    named = [n for _, n in outline if folded(n.title).startswith(folded(title)) and n.page <= page]
    assert named, f"no Section titled {title!r} by p. {page}"
    return max(named, key=lambda n: n.page).id


# Questions: golden.md §8, plus AC3.
golden_rows = [
    [cell.strip() for cell in line.strip().strip("|").split("|")]
    for line in (GOLDEN / "golden.md").read_text().splitlines()
    if re.match(r"\| Q\d+ \|", line)
]
asked = [
    {"id": r[0], "type": r[1], "question": r[2], "expected": r[3], "citation": r[4]}
    for r in golden_rows
] + [
    {
        "id": "AC3",
        "type": "acceptance (#21)",
        "question": "dividende proposé",
        "expected": "The p. 10 Dividende paragraph: 4,00 dirhams per share, 3,5 milliards in all.",
        "citation": "p. 10, Dividende",
    }
]

# Evidence: evidence.csv, plus AC3. Each item keeps the parsed Paragraphs that hold it.
with open(GOLDEN / "evidence.csv", newline="") as f:
    rows = list(csv.DictReader(f))
rows.append({"question": "AC3", "page": "10", "section": "Dividende", "text": "4,00"})
evidence = {}
for row in rows:
    page = int(row["page"])
    evidence.setdefault(row["question"], []).append(
        {
            "page": page,
            "section": row["section"],
            "text": row["text"],
            "section_id": cited_section(row["section"], page),
            "paragraphs": [
                i
                for i, (_, p) in enumerate(paragraphs)
                if page in p.pages and holds(texts[i], row["text"])
            ],
        }
    )
unparsed = [
    (q, it["page"], it["text"]) for q, its in evidence.items() for it in its if not it["paragraphs"]
]
print(f"{len(rows)} evidence items; missing from the parsed text: {unparsed or 'none'}")

# Noise: Docling's own page headers and footers wherever they recur as text, lone headings, and
# fragments too short to answer anything.
furniture = {
    folded(t["text"])
    for t in json.loads(output.read_text())["texts"]
    if t["label"] in ("page_header", "page_footer")
}
noise = [
    folded(t) in furniture or p.kind == "heading" or len(t.strip()) < 20
    for (_, p), t in zip(paragraphs, texts)
]

METHODS = {"full-text": lambda q: bm25(q)}
for m in MODELS:
    METHODS[f"vector {m}"] = lambda q, m=m: vector(q, m)
    METHODS[f"hybrid {m}"] = lambda q, m=m: hybrid(q, m)
for m in MODELS:
    METHODS[f"hybrid-fr {m}"] = lambda q, m=m: hybrid_fr(q, m)
for r in RERANKERS:
    METHODS[f"rerank {r}"] = lambda q, r=r: rerank(q, r)


def score(q, scores):
    order = [int(i) for i in ranks(scores)[:10]]
    top, items = order[:5], evidence[q]
    holding = {i for it in items for i in it["paragraphs"]}
    found = [any(i in it["paragraphs"] for i in top) for it in items]
    in_section = [
        any(it["section_id"] in lineage(paragraphs[i][0].section_id) for i in top) for it in items
    ]
    first = next((rank for rank, i in enumerate(order, 1) if i in holding), None)
    return {
        "top": top,
        "page": any(paragraphs[i][1].page in QUESTIONS[q][1] for i in top),
        "found": found,
        "in_section": in_section,
        "section": sum(in_section) / len(items),
        "evidence": sum(found) / len(items),
        "rr": 1 / first if first else 0.0,
        "noise": sum(noise[i] for i in top),
    }


scored, seconds = {q: {} for q in QUESTIONS}, Counter()
for q, (question, _) in QUESTIONS.items():
    for name, search in METHODS.items():
        started = time.perf_counter()
        scores = search(question)
        seconds[name] += time.perf_counter() - started
        scored[q][name] = score(q, scores)
display(
    pd.DataFrame(
        {
            name: {q: f"{sum(r[name]['found'])}/{len(r[name]['found'])}" for q, r in scored.items()}
            for name in METHODS
        }
    ).rename_axis("evidence found")
)
scorecard = pd.DataFrame(
    {
        name: {
            "page@5": np.mean([r[name]["page"] for r in scored.values()]),
            "section@5": np.mean([r[name]["section"] for r in scored.values()]),
            "evidence@5": np.mean([r[name]["evidence"] for r in scored.values()]),
            "MRR@10": np.mean([r[name]["rr"] for r in scored.values()]),
            "noise@5": np.mean([r[name]["noise"] for r in scored.values()]),
        }
        for name in METHODS
    }
).T
scorecard["seconds per query"] = [seconds[name] / len(QUESTIONS) for name in METHODS]
scorecard.round(2)

In [ ]:
# Q11 and Q12 have no answer in the report. Each method's top score on them, next to its median
# top score on the answerable questions, shows whether a threshold could say "not found".
unanswerable = [q for q in asked if q["id"] not in evidence]
pd.DataFrame(
    {
        name: {
            "answerable, median": np.median(
                [search(QUESTIONS[q][0])[units].max() for q in QUESTIONS]
            ),
            **{q["id"]: search(q["question"])[units].max() for q in unanswerable},
        }
        for name, search in METHODS.items()
    }
).T.round(3)

## What this tells PR B

**Decision: hybrid with `Qwen3-Embedding-0.6B`**, full-text and vector fused by reciprocal rank
fusion, the question as asked. It finds 0.41 of the cited evidence in the top 5, with no noise, in
well under a second per query. evidence@5 and MRR@10 decide: they count the answer itself, where
page@5 only counts its page.

- **Hybrid-fr** finds the right page more often (page@5 0.73 against 0.55) but not more evidence,
  and adds an LLM call to every query.
- **The Qwen3 reranker** scores best (evidence@5 0.47, section@5 0.71) but takes about 30 seconds
  per query on a laptop and lets noise back in (0.18). Worth another look, with a smaller pool,
  once search runs in the app.
- **No method finds Q2, Q5, Q6, Q8 or Q9.** The next gain is in how tables are indexed, not in
  ranking.
- **No score tells the unanswerable questions (Q11, Q12) apart**, so search cannot yet say "not
  found".

The full-text side here is a BM25 stand-in; the app uses Neo4j's `standard-folding` index.

## 6. Error analysis

Writes a page to inspect the pipeline stage by stage: what Docling found on every page and what
looks wrong, each golden question's search results drawn on the report, and every golden figure
(`figures.csv`) checked against the parsed text of its page. The page is a separate tool, not
part of the app; it is written to `data/notebook/error-analysis/`, which stays out of git because
it shows the report.

In [ ]:
OUT = WORK / "error-analysis"
(OUT / "pages").mkdir(parents=True, exist_ok=True)


def numbers_in(answer):
    found = re.findall(r"\d{1,3}(?:\s\d{3})+|\d+(?:,\d+)?%?", answer)
    return [
        n
        for n in dict.fromkeys(found)
        if len(re.sub(r"\D", "", n)) >= 3 and not re.fullmatch(r"20\d\d", n)
    ]


questions = []
for q in asked:
    pages = sorted(QUESTIONS.get(q["id"], (None, set()))[1])
    items = [
        {k: it[k] for k in ("page", "section", "text", "paragraphs")}
        for it in evidence.get(q["id"], [])
    ]
    results = {
        name: scored[q["id"]][name]
        if q["id"] in scored
        else {"top": [int(i) for i in ranks(search(q["question"]))[:5]]}
        for name, search in METHODS.items()
    }
    questions.append(
        {
            **q,
            "pages": pages,
            "numbers": numbers_in(q["expected"]),
            "evidence": items,
            "results": results,
        }
    )

# Figures: is each golden value in the parsed text of its page?
figures = []
with open(GOLDEN / "figures.csv", newline="") as f:
    for row in csv.DictReader(f):
        value, page = row["value_as_written"], int(row["page"])
        where = [i for i, t in enumerate(texts) if contains(t, value)]
        on_page = [i for i in where if paragraphs[i][1].page == page]
        figures.append(
            {
                "line_item": row["line_item"],
                "label": row["raw_label"],
                "period": row["period"],
                "scope": row["basis_scope"],
                "standard": row["basis_standard"],
                "value": value,
                "unit": row["unit_scale"],
                "page": page,
                "table": row["table_title"],
                "row": row["row_label"],
                "column": row["column_label"],
                "notes": row["notes"],
                "on_page": bool(on_page),
                "matches": on_page,
                "elsewhere": [i for i in where if i not in on_page][:20],
            }
        )

# Parsing: every item Docling found, per page, and whether its text reaches a Chunk.
from docling_core.types.doc import DoclingDocument, TableItem

from bloomberg_invest.adapters.parsing.docling_parser import HEADINGS, TEXTS

docling = DoclingDocument.load_from_json(output)
items = {n: [] for n in sizes}
for item, _ in docling.iterate_items():
    for prov in getattr(item, "prov", None) or []:
        box = prov.bbox.to_top_left_origin(page_height=sizes[prov.page_no][1])
        is_table = isinstance(item, TableItem)
        text = item.export_to_markdown(doc=docling) if is_table else getattr(item, "text", "")
        items[prov.page_no].append(
            {
                "label": item.label.value,
                "box": [round(v, 1) for v in (box.l, box.t, box.r, box.b)],
                "text": (text or "") if is_table else (text or "")[:800],
                "used": is_table or item.label in HEADINGS | TEXTS,
            }
        )


def check(item, width, height):
    """What looks wrong with one parsed item, or None."""
    left, top, right, bottom = item["box"]
    share = (right - left) * (bottom - top) / (width * height)
    words = len(item["text"].split())
    if item["label"] in ("picture", "chart") and share > 0.25:
        level = "lost" if share > 0.5 else "warn"
        return {
            "level": level,
            "text": f"A {item['label']} covers {share:.0%} of the page: its text is not read",
        }
    if item["label"] == "code":
        return {"level": "warn", "text": "Text labelled as code"}
    if item["label"] == "section_header" and words > 12 and not item["text"].isupper():
        return {"level": "warn", "text": f"A {words}-word heading: probably a sentence"}
    return None


def flags(number):
    """What looks wrong in a page's parse, the most serious first."""
    width, height = sizes[number]
    found = []
    for item in items[number]:
        item["check"] = check(item, width, height)
        if item["check"]:
            found.append(item["check"])
    if not any(item["used"] for item in items[number]):
        found.append({"level": "warn", "text": "Nothing on this page reaches a Chunk"})
    return sorted(found, key=lambda f: f["level"] != "lost")


# Pages, rendered once: full size for the viewer, small for the contact sheet.
(OUT / "thumbs").mkdir(exist_ok=True)
for number in range(1, len(pdf) + 1):
    image = OUT / "pages" / f"{number}.jpg"
    if not image.exists():
        pdf[number - 1].render(scale=2).to_pil().convert("RGB").save(image, quality=82)
    thumb = OUT / "thumbs" / f"{number}.jpg"
    if not thumb.exists():
        pdf[number - 1].render(scale=0.4).to_pil().convert("RGB").save(thumb, quality=80)

data = {
    "document": {
        "title": "Maroc Telecom · Rapport financier annuel 2025",
        "id": detail.id,
        "parse_run": run.id,
        "parser": f"{run.tool} {run.tool_version}",
        "parsed_at": f"{run.finished_at:%Y-%m-%d %H:%M}",
        "sections": len(outline),
        "chunks": len(chunks),
        "tables": int((table.kind == "table").sum()),
        "paragraphs": len(paragraphs),
        "cap": CAP,
    },
    "methods": list(METHODS),
    "pages": {
        n: {"width": w, "height": h, "flags": flags(n), "items": items[n]}
        for n, (w, h) in sizes.items()
    },
    "paragraphs": [
        {
            "page": p.page,
            "boxes": p.boxes,
            "text": p.text,
            "chunk": c.index,
            "section": titles.get(c.section_id, ""),
            "noise": noise[i],
            "kind": p.kind,
            "unit": bool(units[i]),
            "index": indexed[i],
        }
        for i, (c, p) in enumerate(paragraphs)
    ],
    "questions": questions,
    "figures": figures,
}
template = (ROOT / "backend" / "notebooks" / "error_analysis.html").read_text()
payload = json.dumps(data, ensure_ascii=False).replace("</", "<\\/")
(OUT / "index.html").write_text(template.replace("/*DATA*/null", payload))
missed = sum(not f["on_page"] for f in figures)
flagged = sum(bool(page["flags"]) for page in data["pages"].values())
print(
    f"{len(figures) - missed} of {len(figures)} figures found on their page; {flagged} pages flagged"
)
print((OUT / "index.html").as_uri())